In [1]:
from torchvision.models import mobilenetv3
from torch import nn
from functools import partial
from xaikd import models

/home/pat/projects/xai-kd/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
models.get_untrained_model("student-mobilenets-lastd25", num_classes=10)

PatMobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1

In [3]:
models.get_untrained_model("student-mobilenets-lastd25v2", num_classes=10)

PatMobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1

In [25]:
def _student_very_small_bottleneck(
    num_classes, dim1, dim2, dim3, dim4, **kwargs
) -> nn.Module:
    dilation = 1

    width_mult = 1
    bneck_conf = partial(mobilenetv3.InvertedResidualConfig, width_mult=width_mult)
    adjust_channels = partial(
        mobilenetv3.InvertedResidualConfig.adjust_channels, width_mult=width_mult
    )

    inverted_residual_setting = [
        # same conf as mobiletnet-s
        bneck_conf(16, 3, 16, 16, True, "RE", 2, 1),  # C1
        bneck_conf(16, 3, 72, 24, False, "RE", 2, 1),  # C2
        bneck_conf(24, 3, 88, 24, False, "RE", 1, 1),
        bneck_conf(24, 5, 96, 40, True, "HS", 2, 1),  # C3
        bneck_conf(40, 5, 240, 40, True, "HS", 1, 1),
        bneck_conf(40, 5, 240, 40, True, "HS", 1, 1),
        # modified conf
        bneck_conf(40, 5, 120, dim1, True, "HS", 1, 1),
        bneck_conf(dim1, 5, dim2, dim1, True, "HS", 1, 1),
        bneck_conf(dim1, 5, dim2 * 2, dim1 * 2, True, "HS", 2, dilation),  # C4
        bneck_conf(
            dim1 * 2,
            5,
            dim3,
            dim1 * 2,
            True,
            "HS",
            1,
            dilation,
        ),
        bneck_conf(
            dim1 * 2,
            5,
            dim3,
            dim1*2,
            True,
            "HS",
            1,
            dilation,
        ),
    ]
    last_channel = adjust_channels(dim4)  # C5

    return mobilenetv3._mobilenet_v3(
        inverted_residual_setting,
        last_channel,
        num_classes=num_classes,
        weights=None,
        progress=False,
        **kwargs,
    )

_student_very_small_bottleneck(10, dim1=8, dim2=20, dim3=8, dim4=7).features[-2:]

Sequential(
  (11): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(8, 8, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), groups=8, bias=False)
        (1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (2): SqueezeExcitation(
        (avgpool): AdaptiveAvgPool2d(output_size=1)
        (fc1): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1))
        (fc2): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1))
        (activation): ReLU()
        (scale_activation): Hardsigmoid()
      )
      (3): Conv2dNormActivation(
        (0): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.0

In [24]:
32*6

192

In [16]:
# mobilenetv3.InvertedResidualConfig(
#     input_channels=8,
#     kernel=3,
#     expanded_channels=16,
#     out_channels=7,
#     width_mult=1,
#     use_se=True,
# )

TypeError: InvertedResidualConfig.__init__() missing 4 required positional arguments: 'use_se', 'activation', 'stride', and 'dilation'